In [1]:
from types import SimpleNamespace
import torch as t
import einops
from math import ceil, sqrt

In [2]:
cfg = SimpleNamespace(
    seq_len = 256,
    d_head = 64,
    SRAM_SIZE = 164,   #Streaming Multiprocessing의 shared memory 용량 기준 A100, 164kb
    precision = 2     #2byte = 16bit
)

In [3]:
def FlashAttention(cfg: SimpleNamespace, Q, K, V):
  M = cfg.SRAM_SIZE * 1024 // cfg.precision
  B_c = ceil(M / (4*cfg.d_head))                   #K와 V의 blocksize(몇 개의 토큰을 하나로 묶을 지)
  B_r = min(B_c, cfg.d_head)                       #Q와 O의 blocksize. Warp와 Tensor Core 연산 효율을 위해 가능하면 d_head로
  O = t.zeros((cfg.seq_len, cfg.d_head))                   #attention pattern과 V 벡터가 곱해져 만들어지는 결과값

  cur_max = t.full((cfg.seq_len,), float('-inf'))
  cum_sum = t.zeros((cfg.seq_len,))

  for j in range(ceil(cfg.seq_len/B_c)):
    j_start = j*B_c
    j_end = min(cfg.seq_len, (j+1)*B_c)
    k_block = K[j_start:j_end, :]
    v_block = V[j_start:j_end, :]

    for i in range(ceil(cfg.seq_len/B_r)):
      i_start = i*B_r
      i_end = min(cfg.seq_len, (i+1)*B_r)
      q_block = Q[i_start:i_end , :]
      pattern_block = einops.einsum(q_block, k_block, "q_seq d_head, k_seq d_head -> q_seq k_seq") / sqrt(cfg.d_head)

      max_block = t.max(pattern_block, dim=-1).values
      p_block = t.exp(pattern_block - max_block.unsqueeze(-1))
      sum_block = t.sum(p_block, dim=-1)

      new_max = t.maximum(max_block, cur_max[i_start: i_end])

      alpha = t.exp(cur_max[i_start:i_end] - new_max)
      beta = t.exp(max_block - new_max)

      new_sum = alpha * cum_sum[i_start:i_end] + beta * sum_block

      O[i_start:i_end] = (
          (cum_sum[i_start:i_end] * alpha).unsqueeze(-1) * O[i_start:i_end]
          + beta.unsqueeze(-1) * (p_block @ v_block)
      ) / new_sum.unsqueeze(-1)

      cur_max[i_start: i_end] = new_max
      cum_sum[i_start: i_end] = new_sum

  return O

아래는 claude가 짜준 검증 코드입니다

In [4]:
import torch as t
import torch.nn.functional as F
from types import SimpleNamespace

def naive_attention(Q, K, V):
    S = Q @ K.T / Q.shape[-1] ** 0.5
    return S.softmax(dim=-1) @ V

def check(seq_len=100, d_head=64, sram_mb=0.02, seed=0):
    t.manual_seed(seed)
    cfg = SimpleNamespace(SRAM_SIZE=sram_mb, precision=4,
                          d_head=d_head, seq_len=seq_len)
    Q, K, V = (t.randn(seq_len, d_head) for _ in range(3))

    out_flash = FlashAttention(cfg, Q, K, V)
    out_naive = naive_attention(Q, K, V)
    out_ref = F.scaled_dot_product_attention(Q[None, None], K[None, None], V[None, None])[0, 0]

    print("naive vs ref  :", (out_naive - out_ref).abs().max().item())
    print("flash vs ref  :", (out_flash - out_ref).abs().max().item())
    assert t.allclose(out_flash, out_ref, atol=1e-5, rtol=1e-4)
    print("OK")

check()                                  # 블록이 여러 개 + 나머지 블록
check(seq_len=37, d_head=16)             # 작은 크기
check(seq_len=256, d_head=64, sram_mb=1) # 블록 1개 (경계 케이스)

naive vs ref  : 2.980232238769531e-07
flash vs ref  : 2.682209014892578e-07
OK
naive vs ref  : 2.384185791015625e-07
flash vs ref  : 2.384185791015625e-07
OK
naive vs ref  : 1.1920928955078125e-06
flash vs ref  : 5.960464477539062e-07
OK


In [5]:
def peak_mem(fn, *args):
    t.cuda.empty_cache(); t.cuda.reset_peak_memory_stats()
    base = t.cuda.memory_allocated()
    fn(*args)
    return (t.cuda.max_memory_allocated() - base) / 2**20  # MiB

for N in [512, 1024, 2048, 4096, 8192]:
    Q, K, V = (t.randn(N, 64, device="cuda") for _ in range(3))
    cfg = SimpleNamespace(SRAM_SIZE=0.1, precision=4, d_head=64, seq_len=N)
    print(N, peak_mem(naive_attention, Q, K, V))
    # FlashAttention 안의 t.zeros / t.full에 device=Q.device를 넣어야 GPU에서 돌아가요

512 10.25
1024 8.25
2048 32.5
4096 129.0
8192 514.0
